# 04 — Feature Selection Scenarios

Four **input scenarios** feed the segmentation models:

| key | scenario | channels |
|-----|----------|----------|
| `single_date` | Single date (peak-NDVI, ~14 July), all bands, **no selection** | 10 |
| `mt_ndvi` | Multi-temporal NDVI — 4 dates (per-quarter peak NDVI) × bands | 4×bands |
| `gsi` | **GSI** feature selection, per-crop **normalized score ≥ 0.5** | variable |
| `rf`  | **RF importance ranking**, per-crop **normalized score ≥ 0.5** | variable |

This notebook implements the two selection methods **inline** (the same
algorithm as `stages/selection/{gsi_selection,feature_importance_selection}.py`)
so each step is visible: sample crop pixels → score every (date×band) channel →
per-crop min-max normalize → keep channels ≥ 0.5 → union across crops
(Wei et al. 2023). `save_selection` is imported only to serialize the result in
the schema the training reads.

> Heavy: reads the full processed S2 stack. Needs `data/processed/` locally.

In [2]:
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'cropmap_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from cropmap_pipeline import config as C

In [3]:
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from sklearn.ensemble import RandomForestClassifier

# helpers pulled from config / repo (serialization + input listing only)
from cropmap_pipeline.stages.selection._utils import save_selection
from cropmap_pipeline.stages.selection.band_scoring import get_train_year_inputs

S2_BAND_NAMES   = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
KEEP_CLASSES    = [1, 3, 24, 36, 54, 69, 75, 76]
CDL_CLASS_NAMES = {
    1:  "Corn",
    3:  "Rice",
    24: "Winter Wheat",
    36: "Alfalfa",
    54: "Tomatoes",
    69: "Grapes",
    75: "Almonds",
    76: "Walnuts",
}
SAMPLE_FRACTION = 0.20
S2_NODATA       = -9999.0
RF_N_ESTIMATORS = 500
RF_MAX_PIXELS   = 1_000_000
THRESH          = 0.5   # normalized-score threshold (main setting)

_yr, S2_PATHS, CDL_PATH = get_train_year_inputs()   # v6.1 = 2024
print(len(S2_PATHS), 'S2 files | threshold =', THRESH)

could not write s2_validity_cache.json: [Errno 2] No such file or directory: '/Users/dikaizm/Documents/PROGRAMMING/ml-ai/research-crop-mapping-thesis/research-crop-mapping-geoai/cropmap-remote-sensing-exps/data/processed/s2/2024/s2_validity_cache.json'


AssertionError: No S2 files in /Users/dikaizm/Documents/PROGRAMMING/ml-ai/research-crop-mapping-thesis/research-crop-mapping-geoai/cropmap-remote-sensing-exps/data/processed/s2/2024

## 1. Sample crop pixels

Read CDL once → crop-pixel indices → sample a fraction → read those pixels from
each date's S2 file (one file at a time to bound RAM). Columns = (band × date) channels.

In [ ]:
def build_channel_names(s2_paths):
    names, dates = [], []
    for p in s2_paths:
        m = re.search(r'_(\d{4}_\d{2}_\d{2})(_processed)?\.tif$', os.path.basename(p))
        d = m.group(1).replace('_', '') if m else os.path.basename(p)[:8]
        if d not in dates: dates.append(d)
        names += [f'{b}_{d}' for b in S2_BAND_NAMES]
    return names, sorted(dates)

def sample_pixels(s2_paths, cdl_path, bandnames):
    with rasterio.open(cdl_path) as src:
        lbl = src.read(1).astype(np.int32).flatten()
    valid = np.isin(lbl, KEEP_CLASSES)
    vidx  = np.where(valid)[0]; lblv = lbl[valid]; del lbl
    rng = np.random.default_rng(42)
    n = min(len(vidx), max(1000, int(len(vidx) * SAMPLE_FRACTION)))
    ch = rng.choice(len(vidx), n, replace=False)
    flat = vidx[ch]; lbls = lblv[ch]
    data = np.full((n, len(bandnames)), np.nan, np.float32); col = 0
    for p in s2_paths:
        with rasterio.open(p) as src:
            arr = src.read().astype(np.float32)
        arr[arr == S2_NODATA] = np.nan
        a2 = arr.reshape(arr.shape[0], -1).T; del arr
        data[:, col:col + a2.shape[1]] = a2[flat]; col += a2.shape[1]
    df = pd.DataFrame(data, columns=bandnames)
    df.insert(0, 'class_label', lbls.astype(int))
    return df

bandnames, dates = build_channel_names(S2_PATHS)
df = sample_pixels(S2_PATHS, CDL_PATH, bandnames)
print(f'sampled {df.shape[0]:,} px × {len(bandnames)} channels over {len(dates)} dates')

## 2. GSI selection

**Global Separability Index** (Li et al. 2023). For crop *s* vs each other crop *o*,
per channel: $SI_{s,o}=|\mu_s-\mu_o|/(1.96(\sigma_s+\sigma_o))$; then $GSI_s$ = mean over *o*.
Per crop: min-max normalize to [0,1], keep channels ≥ threshold.

In [ ]:
def gsi_per_crop(df, bandnames):
    x = df[bandnames].values.astype(np.float32); y = df['class_label'].values
    mu, std, valid = {}, {}, []
    for c in KEEP_CLASSES:
        m = y == c
        if m.sum() < 10: continue
        mu[c] = np.nanmean(x[m], 0); std[c] = np.nanstd(x[m], 0); valid.append(c)
    gsi = {}
    for s in KEEP_CLASSES:
        if s not in valid:
            gsi[s] = pd.Series(0.0, index=bandnames); continue
        pairs = [np.abs(mu[s] - mu[o]) / (1.96 * (std[s] + std[o]) + 1e-9)
                 for o in valid if o != s]
        gsi[s] = pd.Series(np.mean(pairs, 0).astype(np.float32), index=bandnames)
    return gsi

def threshold_select(scores_per_crop, thr):
    """Per-crop min-max normalize → keep channels with norm score >= thr."""
    sel = {}
    for c in KEEP_CLASSES:
        s = scores_per_crop[c]; lo, hi = float(s.min()), float(s.max())
        norm = (s - lo) / (hi - lo) if hi > lo else pd.Series(0.0, index=s.index)
        sel[c] = norm[norm >= thr].sort_values(ascending=False).index.tolist()
    return sel

gsi_scores = gsi_per_crop(df, bandnames)
gsi_sel = threshold_select(gsi_scores, THRESH)
for c in KEEP_CLASSES:
    print(f'  {CDL_CLASS_NAMES[c]:14s}: {len(gsi_sel[c]):3d} ch  top-3 {gsi_sel[c][:3]}')

## 3. RF importance selection

**One** multi-class RandomForest; per-crop importance via class-conditional MDI —
decompose each split's Gini decrease weighted by that class's share at the node
(Wei et al. 2023). Then the same per-crop normalize → threshold.

In [ ]:
def train_rf(df, bandnames, seed=42):
    x = df[bandnames].values.astype(np.float32); y = df['class_label'].values.astype(int)
    if RF_MAX_PIXELS and len(y) > RF_MAX_PIXELS:
        idx = np.random.default_rng(seed).choice(len(y), RF_MAX_PIXELS, replace=False)
        x, y = x[idx], y[idx]
    med = np.nanmedian(x, 0); x = np.where(np.isnan(x), med, x)
    rf = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, class_weight='balanced',
                                n_jobs=-1, random_state=seed)
    rf.fit(x, y)
    return rf

def rf_per_crop(rf, bandnames):
    classes = list(rf.classes_); nfe = len(bandnames); nc = len(classes)
    imp = np.zeros((nc, nfe))
    for tree in rf.estimators_:
        t = tree.tree_; feat = t.feature; ns = t.n_node_samples
        val = t.value[:, 0, :]; gini = t.impurity; cl = t.children_left; cr = t.children_right
        root = val[0]; N = ns[0]
        for nid in range(t.node_count):
            if cl[nid] == -1 or feat[nid] < 0: continue
            l, r = cl[nid], cr[nid]; np_, nl, nr = ns[nid], ns[l], ns[r]
            dg = gini[nid] - (nl / np_) * gini[l] - (nr / np_) * gini[r]
            if dg <= 0: continue
            w = (np_ / N) * dg
            for ci in range(nc):
                if root[ci] == 0: continue
                imp[ci, feat[nid]] += (val[nid, ci] / root[ci]) * w
    imp /= len(rf.estimators_)
    for ci in range(nc):
        s = imp[ci].sum()
        if s > 0: imp[ci] /= s
    c2i = {c: i for i, c in enumerate(classes)}
    return {c: pd.Series(imp[c2i[c]].astype(np.float32), index=bandnames)
            if c in c2i else pd.Series(0.0, index=bandnames) for c in KEEP_CLASSES}

rf = train_rf(df, bandnames)
rf_scores = rf_per_crop(rf, bandnames)
rf_sel = threshold_select(rf_scores, THRESH)
for c in KEEP_CLASSES:
    print(f'  {CDL_CLASS_NAMES[c]:14s}: {len(rf_sel[c]):3d} ch  top-3 {rf_sel[c][:3]}')

## 4. Save + compare

Union across crops, write the train-compatible JSON (`select_{gsi,rf}_direct_s0.5.json`),
and compare the two channel sets. Expect GSI broad/year-spread, RF concentrated.

In [ ]:
def union(sel):
    seen = []
    for chans in sel.values():
        for ch in chans:
            if ch not in seen: seen.append(ch)
    return seen

gsi_u, rf_u = union(gsi_sel), union(rf_sel)
for sel, name in [(gsi_sel, 'gsi_direct'), (rf_sel, 'rf_direct')]:
    stem = f'select_{name}_s{THRESH:g}'
    save_selection(sel, C.PROCESSED_DIR / f'{stem}.json', C.PROCESSED_DIR / f'{stem}_bands.txt',
                   selector=name, score_threshold=THRESH,
                   meta={'years': [_yr], 'note': 'computed inline in notebook 04'})
    print('wrote', C.PROCESSED_DIR / f'{stem}.json')

gs, rs = set(gsi_u), set(rf_u)
print(f'\nGSI {len(gs)} channels | RF {len(rs)} channels')
print(f'shared {len(gs & rs)} | GSI-only {len(gs - rs)} | RF-only {len(rs - gs)}')
plt.figure(figsize=(5, 4))
plt.bar(['GSI', 'RF'], [len(gs), len(rs)], color=['steelblue', 'darkorange'])
plt.ylabel('# channels (norm score ≥ 0.5)'); plt.title('Selected channel count')
plt.tight_layout(); plt.show()